In [ ]:
from collections import defaultdict

def apriori(transactions, min_sup):
    """
    Finds frequent itemsets using the Apriori algorithm.

    Args:
        transactions: A list of transactions, where each transaction is a set of items.
        min_sup: The minimum support threshold (a float between 0 and 1).

    Returns:
        A dictionary where keys are the sizes of frequent itemsets (k) and
        values are sets of frequent k-itemsets.
    """

    num_transactions = len(transactions)

    # 1. Frequent 1-itemsets
    item_counts = defaultdict(int)
    for transaction in transactions:
        for item in transaction:
            item_counts[item] += 1

    frequent_1_itemsets = {
    frozenset([item]) for item, count in item_counts.items() if count / num_transactions >= min_sup
}

    frequent_itemsets = {1: frequent_1_itemsets}
    k = 2

    while True:
        # 2. Generate candidate k-itemsets
        candidate_k_itemsets = set()
        for itemset1 in frequent_itemsets[k - 1]:
            for itemset2 in frequent_itemsets[k - 1]:
                if len(itemset1.union(itemset2)) == k: #Join condition
                    candidate_k_itemsets.add(frozenset(itemset1.union(itemset2))) #Use frozenset for hashing

        if not candidate_k_itemsets:  # No more candidates
            break

        # 3. Prune infrequent k-itemsets
        itemset_counts = defaultdict(int)
        for transaction in transactions:
            for itemset in candidate_k_itemsets:
                if itemset.issubset(transaction):
                    itemset_counts[itemset] += 1

        frequent_k_itemsets = {
            itemset for itemset, count in itemset_counts.items()
            if count / num_transactions >= min_sup
        }

        if not frequent_k_itemsets:  # No more frequent k-itemsets
            break

        frequent_itemsets[k] = frequent_k_itemsets
        k += 1

    return frequent_itemsets

def generate_association_rules(frequent_itemsets, min_conf):
    """Generates association rules from frequent itemsets."""
    rules = []
    for k, itemsets in frequent_itemsets.items():
        if k > 1:  # Rules only from itemsets of size 2 or more
            for itemset in itemsets:
                for i in range(1, k):
                    antecedent = frozenset(list(itemset)[:i])
                    consequent = itemset - antecedent
                    confidence = support(itemset, transactions) / support(antecedent, transactions) #Helper function below
                    if confidence >= min_conf:
                        rules.append((antecedent, consequent, confidence))
    return rules

def support(itemset, transactions):
    """Calculates the support of an itemset."""
    count = 0
    for transaction in transactions:
        if itemset.issubset(transaction):
            count += 1
    return count / len(transactions)


# Example usage (using your sample data):
transactions = [
    {"Milk", "Bread", "Diapers"},
    {"Milk", "Diapers", "Egg"},
    {"Milk", "Bread", "Diapers"},
    {"Bread", "Diapers", "Egg"},
    {"Milk", "Bread", "Egg"},
]

min_sup = 0.4
min_conf = 0.5 #Example minimum confidence

frequent_itemsets = apriori(transactions, min_sup)

for k, itemsets in frequent_itemsets.items():
    print(f"Frequent {k}-itemsets: {itemsets}")

rules = generate_association_rules(frequent_itemsets, min_conf)
print("\nAssociation Rules:")
for antecedent, consequent, confidence in rules:
    print(f"{antecedent}-> {consequent} (Confidence: {confidence:.2f})")
